# 🎵 ACE-Step v1.5 **base** — Entraînement LoRA / LoKR avec Side-Step (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/walidb27/ACE-Step-1.5/blob/claude/nice-volta-wvnuc6/notebooks/ACE_Step_v15_Base_SideStep_Colab.ipynb)

L'interface Gradio d'ACE-Step n'entraîne que le modèle **turbo** (timesteps discrets « shift=3 »).
Ce notebook utilise **Side-Step** (`train.py`, mode `fixed`, intégré au dépôt) qui reproduit le
vrai schéma d'entraînement des modèles **base / sft** :

- timesteps **continus** (logit-normal, paramètres lus dans le `config.json` du modèle) ;
- **CFG dropout** (le modèle base a été entraîné avec la guidance sans classifieur) ;
- LoRA (PEFT) ou LoKR (LyCORIS), gradient checkpointing, optimiseurs 8-bit / Prodigy.

**Étapes :** 1) GPU & Drive → 2) installation → 3) téléchargement des poids → 4) dataset →
5) prétraitement → 6) entraînement → 7) récupération de l'adaptateur.

> 💡 GPU conseillé : **L4 ou A100** (bf16). Le T4 fonctionne (fp16, ~7 Go de VRAM avec gradient
> checkpointing) mais il est plus lent et le fp16 est moins stable.

## 1. GPU et Google Drive

Tout est rangé dans votre dossier ACE-Step du Drive, comme le ferait l'interface Gradio :

```
MyDrive/ACE-Step/
├── checkpoints/                    poids (réutilisés, seul le manquant est téléchargé)
├── datasets/maes2/                 vos audios (+ JSON éventuel)
├── datasets/preprocessed_tensors/  tenseurs Side-Step (suffixe _base)
└── lora_output/<RUN_NAME>/
    ├── checkpoints/epoch_N/        checkpoints intermédiaires
    └── final/                      adaptateur final
```

In [ ]:
#@title Vérifier le GPU, monter le Drive et définir les dossiers
ACESTEP_DRIVE_DIR = "/content/drive/MyDrive/ACE-Step"  #@param {type:"string"}
#@markdown Votre installation ACE-Step sur le Drive.

import os
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "⚠️ Aucun GPU détecté : Exécution > Modifier le type d'exécution > GPU")

from google.colab import drive
drive.mount("/content/drive")

if not os.path.isdir(ACESTEP_DRIVE_DIR):
    raise FileNotFoundError(f"Dossier introuvable : {ACESTEP_DRIVE_DIR}")
WORKDIR = ACESTEP_DRIVE_DIR
CKPT_DIR = os.path.join(WORKDIR, "checkpoints")
TENSORS_ROOT = os.path.join(WORKDIR, "datasets", "preprocessed_tensors")
OUTPUTS_ROOT = os.path.join(WORKDIR, "lora_output")
for d in (CKPT_DIR, TENSORS_ROOT, OUTPUTS_ROOT):
    os.makedirs(d, exist_ok=True)

print("ACE-Step (Drive) :", WORKDIR)
print("Contenu de checkpoints/ :", sorted(os.listdir(CKPT_DIR)) or "(vide)")

## 2. Installation (ACE-Step + Side-Step)

Pip affichera probablement `ERROR: pip's dependency resolver does not currently take into account…`
suivi de conflits sur `cudf`, `cuml`, `cuda-python`, `google-adk`, `pytensor`, `fasthtml`…
**C'est sans conséquence** : ce sont des paquets préinstallés par Colab que ce notebook n'utilise pas.
Ce qui compte, ce sont les dernières lignes : `torch … | CUDA True` puis `✅ Décodage audio OK`.

In [ ]:
#@title Récupérer le code et installer les dépendances (~5 min)
CODE_SOURCE = "clone"  #@param ["clone", "drive"]
#@markdown `clone` (recommandé) : copie fraîche du code dans la VM, rapide, avec Side-Step à jour.
#@markdown `drive` : utilise le code de `MyDrive/ACE-Step` (il doit contenir `train.py` et
#@markdown `acestep/training_v2/`). Plus lent : Python lit le code à travers le Drive.
REPO_URL = "https://github.com/walidb27/ACE-Step-1.5.git"  #@param {type:"string"}
REPO_BRANCH = "claude/nice-volta-wvnuc6"  #@param {type:"string"}

import os

if CODE_SOURCE == "drive":
    REPO_DIR = WORKDIR
    for needed in ("train.py", "acestep/training_v2/cli/train_fixed.py", "requirements.txt"):
        if not os.path.exists(os.path.join(REPO_DIR, needed)):
            raise FileNotFoundError(
                f"{needed} absent de {REPO_DIR} : version trop ancienne, utilisez CODE_SOURCE = clone"
            )
else:
    REPO_DIR = "/content/ACE-Step-1.5"
    if not os.path.isdir(REPO_DIR):
        !git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# requirements.txt du dépôt, sans flash-attn (compilation de plusieurs dizaines de minutes ;
# Side-Step bascule automatiquement sur SDPA).
!grep -v -i -E "flash-attn|torchcodec" requirements.txt > /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt
# torchcodec (décodage mp3/wav) doit correspondre à torch 2.10 : la dernière version
# ne se charge pas ("undefined symbol") et tout le prétraitement échoue.
!pip install -q "torchcodec==0.10.0"
# Dépendances Side-Step (CLI) + optimiseurs optionnels (AdamW8bit, Prodigy)
!pip install -q rich bitsandbytes prodigyopt

# Vérification finale : doit afficher "CUDA True" puis "Décodage audio OK"
!python -c "import torch, torchaudio; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| bf16', torch.cuda.is_available() and torch.cuda.is_bf16_supported())"
check = !python -c "from torchcodec.decoders import AudioDecoder; import torchcodec; print('OK', torchcodec.__version__)" 2>&1
if check and check[-1].startswith("OK"):
    print("✅ Décodage audio OK (torchcodec", check[-1].split()[-1] + ")")
else:
    print("\n".join(check[-15:]))
    raise RuntimeError("torchcodec ne se charge pas : le prétraitement échouerait. Envoyez la sortie ci-dessus.")

## 3. Vérifier les poids du modèle

Cette cellule regarde ce que contient `MyDrive/ACE-Step/checkpoints/` et ne télécharge **que ce qui
manque**. Si ACE-Step et `acestep-v15-base` sont déjà installés, elle ne télécharge rien.

| Élément | Rôle |
| :--- | :--- |
| `vae/` | convertit l'audio en données pour le modèle |
| `Qwen3-Embedding-0.6B/` | encode le texte (légendes, paroles) |
| `silence_latent.pt` | petit fichier de référence (dans le dossier turbo) |
| `acestep-v15-base/` | le modèle à entraîner |

Elle remet aussi à jour les fichiers de code `.py` du dossier `acestep-v15-base` (comme Gradio).

**`HF_TOKEN` : laissez vide.** Ce n'est utile que pour un modèle privé ; ceux d'ACE-Step sont publics.

In [ ]:
#@title Vérifier / télécharger les poids manquants
HF_TOKEN = ""  #@param {type:"string"}
#@markdown Laisser vide (modèles publics).

import os
from huggingface_hub import snapshot_download

token = HF_TOKEN or None
WEIGHT_FILES = ("model.safetensors", "model.safetensors.index.json",
                "diffusion_pytorch_model.safetensors", "pytorch_model.bin")


def _has_weights(sub):
    """Return True when checkpoints/<sub> already holds model weights."""
    d = os.path.join(CKPT_DIR, sub)
    return os.path.isdir(d) and any(os.path.exists(os.path.join(d, f)) for f in WEIGHT_FILES)


def _has_silence_latent():
    """Return True when a silence_latent.pt exists in any official DiT folder."""
    return any(os.path.isfile(os.path.join(CKPT_DIR, d, "silence_latent.pt"))
               for d in os.listdir(CKPT_DIR) if d.startswith("acestep-v15-"))


patterns = [f"{sub}/*" for sub in ("vae", "Qwen3-Embedding-0.6B") if not _has_weights(sub)]
if not _has_silence_latent():
    patterns.append("acestep-v15-turbo/silence_latent.pt")
if patterns:
    print("Téléchargement depuis ACE-Step/Ace-Step1.5 :", patterns)
    snapshot_download(repo_id="ACE-Step/Ace-Step1.5", local_dir=CKPT_DIR,
                      allow_patterns=patterns, token=token)
else:
    print("✅ VAE, encodeur texte et silence_latent déjà présents")

if _has_weights("acestep-v15-base"):
    print("✅ acestep-v15-base déjà présent :", os.path.join(CKPT_DIR, "acestep-v15-base"))
    # Side-Step charge le code Python rangé dans le dossier du modèle (trust_remote_code).
    # On l'aligne sur le code du dépôt, comme le fait Gradio à chaque démarrage.
    import sys
    sys.path.insert(0, REPO_DIR)
    from acestep.model_downloader import _check_code_mismatch, _sync_model_code_files
    if _check_code_mismatch("acestep-v15-base", CKPT_DIR):
        print("   Code du modèle mis à jour :", _sync_model_code_files("acestep-v15-base", CKPT_DIR))
else:
    cmd = f"python -m acestep.model_downloader --model acestep-v15-base --skip-main --dir {CKPT_DIR}"
    if token:
        cmd += f" --token {token}"
    !{cmd}

!ls {CKPT_DIR} {CKPT_DIR}/acestep-v15-base

## 4. Dataset

Placez vos fichiers audio (`.wav .mp3 .flac .ogg .opus .m4a`, idéalement 30 s à 4 min, **5 morceaux
minimum**) dans `AUDIO_DIR` (sur le Drive) ou envoyez un `.zip` avec la cellule ci-dessous.

Deux façons de fournir les métadonnées :
- **Construire le JSON ici** (cellule 4b) : mot déclencheur (`custom_tag`) en
  **prepend / append / replace**, `genre_ratio`, légendes et paroles par fichier ;
- **Utiliser un JSON existant**, par exemple celui exporté par l'onglet *Dataset* de Gradio
  (qui sait aussi légender automatiquement avec le LM). La cellule 4b peut alors juste
  réappliquer le tag et sa position.

Voir `docs/sidestep/Dataset Preparation.md`.

In [ ]:
#@title Choisir le dossier audio (et éventuellement le JSON)
DATASET_NAME = "maes2"  #@param {type:"string"}
#@markdown Sous-dossier de `MyDrive/ACE-Step/datasets/`.
DATASET_JSON = ""  #@param {type:"string"}
#@markdown Optionnel. Vide = détection automatique d'un JSON ACE-Step (`datasets/maes2.json`
#@markdown ou un `.json` avec une liste `samples` dans le dossier).
UPLOAD_ZIP = False  #@param {type:"boolean"}
#@markdown Cocher pour envoyer un .zip d'audios depuis votre ordinateur.

import json
import os
import zipfile

AUDIO_DIR = os.path.join(WORKDIR, "datasets", DATASET_NAME)
os.makedirs(AUDIO_DIR, exist_ok=True)


def _is_dataset_json(path):
    # True for an ACE-Step dataset JSON (a list, or a dict with "samples").
    try:
        with open(path, encoding="utf-8") as fh:
            data = json.load(fh)
    except (OSError, ValueError):
        return False
    return isinstance(data, list) or (isinstance(data, dict) and "samples" in data)


if not DATASET_JSON:
    candidates = [os.path.join(WORKDIR, "datasets", f"{DATASET_NAME}.json")]
    candidates += sorted(
        os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR)
        if f.endswith(".json") and not f.endswith(".sidestep.json")
    )
    found_json = [c for c in candidates if os.path.isfile(c) and _is_dataset_json(c)]
    if found_json:
        DATASET_JSON = found_json[0]
        print("JSON détecté :", DATASET_JSON)

if UPLOAD_ZIP:
    from google.colab import files
    for name in files.upload():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(name) as zf:
                zf.extractall(AUDIO_DIR)
            os.remove(name)

EXTS = (".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a")
found = [os.path.join(r, f) for r, _, fs in os.walk(AUDIO_DIR) for f in fs if f.lower().endswith(EXTS)]
print(f"{len(found)} fichier(s) audio dans {AUDIO_DIR}")
for f in found[:20]:
    print(" -", os.path.relpath(f, AUDIO_DIR))
if DATASET_JSON and not os.path.isfile(DATASET_JSON):
    raise FileNotFoundError(f"JSON introuvable : {DATASET_JSON}")
if not found and not DATASET_JSON:
    print("⚠️ Aucun audio trouvé : ajoutez des fichiers avant le prétraitement.")

### 4b. Construire le JSON du dataset (tag en prepend / append / replace)

Pour chaque audio `chanson.mp3`, la cellule lit, s'ils existent, des fichiers posés à côté :

| Fichier | Contenu |
| :--- | :--- |
| `chanson.caption.txt` | légende (description du style, des instruments…) |
| `chanson.lyrics.txt` (ou `chanson.txt`) | paroles avec `[Verse]`, `[Chorus]`… |
| `chanson.json` | champs libres : `genre`, `bpm`, `keyscale`, `timesignature`, `language`, `caption`, `lyrics`, `prompt_override`… |

Sans légende, `DEFAULT_CAPTION` est utilisée (ou le nom du fichier si elle est vide).
Sans paroles : `[Instrumental]`. La durée est mesurée automatiquement.

**Position du tag** (ex. tag `mystyle`, légende `dark synthwave`) :
`prepend` → `mystyle, dark synthwave` · `append` → `dark synthwave, mystyle` ·
`replace` → `mystyle` (la légende est ignorée).

À l'inférence, mettez le tag dans le prompt pour activer le style. Toute modification ici
relance automatiquement le prétraitement à l'étape 5.

In [ ]:
#@title Générer le JSON du dataset
BUILD_JSON = True  #@param {type:"boolean"}
#@markdown Décocher pour utiliser tel quel le `DATASET_JSON` choisi plus haut.
CUSTOM_TAG = ""  #@param {type:"string"}
#@markdown Mot déclencheur, ex. `mystyle` (vide = pas de tag).
TAG_POSITION = "prepend"  #@param ["prepend", "append", "replace"]
GENRE_RATIO = 0  #@param {type:"slider", min:0, max:100, step:5}
#@markdown % des morceaux qui utilisent le champ `genre` au lieu de la légende (s'il est renseigné).
DEFAULT_CAPTION = ""  #@param {type:"string"}
DEFAULT_GENRE = ""  #@param {type:"string"}
ALL_INSTRUMENTAL = False  #@param {type:"boolean"}
#@markdown Cocher pour forcer `[Instrumental]` partout (paroles ignorées).

import datetime
import json
import os
import subprocess
import sys
import uuid

import soundfile as sf

sys.path.insert(0, REPO_DIR)
from acestep.training_v2.preprocess_prompt import build_simple_prompt


def _read_text(path):
    """Return stripped file content, or "" when the file is missing."""
    if os.path.isfile(path):
        with open(path, encoding="utf-8") as fh:
            return fh.read().strip()
    return ""


def _duration(path):
    """Measure audio duration in seconds (soundfile, then ffprobe)."""
    try:
        return int(sf.info(path).duration)
    except Exception:
        out = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration",
             "-of", "default=nw=1:nk=1", path],
            capture_output=True, text=True,
        ).stdout.strip()
        return int(float(out)) if out else 0


def _sample_from_files(audio_path):
    """Build one sample dict from an audio file and its sidecar files."""
    base = os.path.splitext(audio_path)[0]
    extra = {}
    if os.path.isfile(base + ".json"):
        with open(base + ".json", encoding="utf-8") as fh:
            extra = json.load(fh)
    stem = os.path.basename(base)
    caption = (_read_text(base + ".caption.txt") or extra.get("caption")
               or DEFAULT_CAPTION or stem.replace("_", " ").replace("-", " "))
    lyrics = (_read_text(base + ".lyrics.txt") or _read_text(base + ".txt")
              or extra.get("lyrics") or "[Instrumental]")
    sample = {
        "id": uuid.uuid4().hex[:8],
        "audio_path": os.path.relpath(audio_path, AUDIO_DIR),
        "filename": os.path.basename(audio_path),
        "caption": caption,
        "genre": extra.get("genre") or DEFAULT_GENRE,
        "lyrics": lyrics,
        "bpm": extra.get("bpm"),
        "keyscale": extra.get("keyscale", ""),
        "timesignature": extra.get("timesignature", ""),
        "duration": extra.get("duration") or _duration(audio_path),
        "language": extra.get("language", "unknown"),
        "prompt_override": extra.get("prompt_override"),
    }
    return sample


def _finalize(sample):
    """Apply instrumental flag and trigger tag to a sample (in place)."""
    if ALL_INSTRUMENTAL:
        sample["lyrics"] = "[Instrumental]"
    sample["is_instrumental"] = sample.get("lyrics", "").strip() == "[Instrumental]"
    if CUSTOM_TAG:
        # Per-sample tags win over the dataset-level one, so set both.
        sample["custom_tag"] = CUSTOM_TAG
    sample["labeled"] = True
    return sample


if BUILD_JSON:
    if DATASET_JSON:
        # Re-tag an existing JSON (e.g. exported by the Gradio Dataset Builder)
        with open(DATASET_JSON, encoding="utf-8") as fh:
            src = json.load(fh)
        samples = src if isinstance(src, list) else src.get("samples", [])
        src_dir = os.path.dirname(os.path.abspath(DATASET_JSON))
        by_name = {os.path.basename(f): f for f in found}
        missing = []
        for s in samples:
            ap = (s.get("audio_path") or s.get("filename", "")).replace("\\", "/")
            full = ap if os.path.isabs(ap) else os.path.join(src_dir, ap)
            if not os.path.isfile(full):
                # JSON built on another machine (e.g. C:/...): match by file name
                full = by_name.get(os.path.basename(ap), "")
            if full:
                s["audio_path"] = os.path.relpath(full, AUDIO_DIR)
                s["filename"] = os.path.basename(full)
            else:
                missing.append(ap)
        if missing:
            print(f"⚠️ {len(missing)} audio(s) du JSON introuvable(s) dans {AUDIO_DIR} :", missing[:5])
        samples = [s for s in samples if s.get("filename") in by_name]
    else:
        samples = [_sample_from_files(p) for p in sorted(found)]
    samples = [_finalize(s) for s in samples]

    dataset = {
        "metadata": {
            "name": DATASET_NAME,
            "custom_tag": CUSTOM_TAG,
            "tag_position": TAG_POSITION,
            "genre_ratio": int(GENRE_RATIO),
            "all_instrumental": all(s["is_instrumental"] for s in samples),
            "num_samples": len(samples),
            "created_at": datetime.datetime.now().isoformat(),
        },
        "samples": samples,
    }
    out_json = os.path.join(AUDIO_DIR, f"{DATASET_NAME}.sidestep.json")
    with open(out_json, "w", encoding="utf-8") as fh:
        json.dump(dataset, fh, ensure_ascii=False, indent=2)
    DATASET_JSON = out_json
    print(f"✅ {len(samples)} morceau(x) -> {out_json}")
    if GENRE_RATIO and not any(s.get("genre") for s in samples):
        print("⚠️ GENRE_RATIO > 0 mais aucun genre renseigné : la légende sera utilisée.")

    # Preview of the exact text prompts Side-Step will encode
    for s in samples[:3]:
        caption_block = build_simple_prompt(s, tag_position=TAG_POSITION).split("# Caption")[1]
        print(f"\n--- {s['filename']} ---{caption_block.split('# Metas')[0].rstrip()}")
else:
    print("JSON utilisé tel quel :", DATASET_JSON or "(aucun : légendes = noms de fichiers)")

## 5. Prétraitement (audio → tenseurs `.pt`)

Deux passes à faible VRAM : VAE + encodeur texte, puis encodeur de conditionnement du DiT **base**.
Les tenseurs doivent être produits **avec le même modèle que celui qu'on entraîne** (ici `base`).
À ne refaire que si le dataset change.

In [ ]:
#@title Lancer le prétraitement
MAX_DURATION = 240  #@param {type:"number"}
FORCE_REPROCESS = False  #@param {type:"boolean"}

%cd {REPO_DIR}
import hashlib
import os
import shlex
import shutil

# Suffixe _base : des tenseurs faits pour turbo (ex. par Gradio) ne conviennent pas à base.
TENSOR_DIR = os.path.join(TENSORS_ROOT, f"{DATASET_NAME}_base")
STAMP = os.path.join(TENSOR_DIR, "_notebook_source.sha1")

# Fingerprint of what the tensors are built from: JSON content + audio list + max duration.
# Side-Step skips existing .pt files, so a changed tag/caption needs a clean folder.
h = hashlib.sha1(f"{AUDIO_DIR}|{MAX_DURATION}|{sorted(found)}".encode())
if DATASET_JSON:
    with open(DATASET_JSON, "rb") as fh:
        h.update(fh.read())
fingerprint = h.hexdigest()
previous = open(STAMP).read().strip() if os.path.isfile(STAMP) else None


def _tensor_count():
    # Finished .pt tensors only (Side-Step also writes .tmp.pt intermediates).
    if not os.path.isdir(TENSOR_DIR):
        return 0
    return sum(f.endswith(".pt") and not f.endswith(".tmp.pt") for f in os.listdir(TENSOR_DIR))


up_to_date = _tensor_count() > 0 and previous == fingerprint

if os.path.isdir(TENSOR_DIR) and (FORCE_REPROCESS or not up_to_date):
    if previous and previous != fingerprint:
        print("Dataset modifié (tag, légendes, fichiers…) : nouveau prétraitement.")
    shutil.rmtree(TENSOR_DIR)
    up_to_date = False

if up_to_date:
    print("Tenseurs à jour :", TENSOR_DIR, "(cocher FORCE_REPROCESS pour refaire)")
else:
    args = [
        "python", "-m", "acestep.training_v2.cli.train_fixed", "--plain",
        "--checkpoint-dir", CKPT_DIR,
        "--model-variant", "base",
        "--preprocess",
        "--audio-dir", AUDIO_DIR,
        "--tensor-output", TENSOR_DIR,
        "--max-duration", str(MAX_DURATION),
    ]
    if DATASET_JSON:
        args += ["--dataset-json", DATASET_JSON]
    cmd = shlex.join(args)
    print(cmd)
    !{cmd}
    if _tensor_count() > 0:
        with open(STAMP, "w") as fh:
            fh.write(fingerprint)

print(f"{_tensor_count()} tenseur(s) dans {TENSOR_DIR}")

## 6. Entraînement Side-Step sur le modèle base

Valeurs par défaut = preset `recommended` de Side-Step. `shift=1.0` et `50` steps sont enregistrés
comme métadonnées d'inférence (valeurs correctes pour base) ; ils n'influencent pas l'entraînement.

Repères :
- **VRAM juste ?** `OFFLOAD_ENCODER`, `OPTIMIZER = adamw8bit`, rang plus bas.
- **Sur-apprentissage** (la sortie recopie le dataset) : moins d'époques, rang plus bas, plus de données.
- **Pas d'effet audible** : plus d'époques ou rang plus élevé.

In [ ]:
#@title Paramètres d'entraînement
RUN_NAME = "maes2_base"  #@param {type:"string"}
ADAPTER_TYPE = "lora"  #@param ["lora", "lokr"]
RANK = 64  #@param {type:"integer"}
ALPHA = 128  #@param {type:"integer"}
DROPOUT = 0.1  #@param {type:"number"}
ATTENTION_TYPE = "both"  #@param ["both", "self", "cross"]
TARGET_LAYERS = "attention"  #@param ["attention", "attention + mlp"]
#@markdown `attention + mlp` adapte aussi les couches feed-forward du DiT : plus de capacité
#@markdown (timbre, détails), adaptateur ~2-3x plus gros, un peu plus de VRAM. Expérimental.
LEARNING_RATE = 1e-4  #@param {type:"number"}
EPOCHS = 100  #@param {type:"integer"}
BATCH_SIZE = 1  #@param {type:"integer"}
GRADIENT_ACCUMULATION = 4  #@param {type:"integer"}
WARMUP_STEPS = 100  #@param {type:"integer"}
OPTIMIZER = "adamw"  #@param ["adamw", "adamw8bit", "adafactor", "prodigy"]
SCHEDULER = "cosine"  #@param ["cosine", "cosine_restarts", "linear", "constant", "constant_with_warmup"]
CFG_RATIO = 0.15  #@param {type:"number"}
SAVE_EVERY = 10  #@param {type:"integer"}
PRECISION = "auto"  #@param ["auto", "bf16", "fp16", "fp32"]
OFFLOAD_ENCODER = False  #@param {type:"boolean"}
SEED = 42  #@param {type:"integer"}
#@markdown ---
#@markdown **Reprise** : chemin d'un checkpoint, ex.
#@markdown `/content/drive/MyDrive/ACE-Step/lora_output/maes2_base/checkpoints/epoch_50_loss_0.1234`
#@markdown (la section 7 liste les checkpoints disponibles).
RESUME_FROM = ""  #@param {type:"string"}

import os
import shlex

OUTPUT_DIR = os.path.join(OUTPUTS_ROOT, RUN_NAME)
print("Checkpoints et adaptateur final ->", OUTPUT_DIR)
if OPTIMIZER == "prodigy" and LEARNING_RATE < 0.1:
    print("ℹ️ Prodigy ajuste le LR lui-même : un LR de 1.0 est conseillé.")

train_args = [
    "python", "train.py", "--plain", "--yes", "fixed",
    "--checkpoint-dir", CKPT_DIR,
    "--model-variant", "base",
    "--dataset-dir", TENSOR_DIR,
    "--output-dir", OUTPUT_DIR,
    "--adapter-type", ADAPTER_TYPE,
    "--attention-type", ATTENTION_TYPE,
    "--lr", str(LEARNING_RATE),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--gradient-accumulation", str(GRADIENT_ACCUMULATION),
    "--warmup-steps", str(WARMUP_STEPS),
    "--optimizer-type", OPTIMIZER,
    "--scheduler-type", SCHEDULER,
    "--cfg-ratio", str(CFG_RATIO),
    "--save-every", str(SAVE_EVERY),
    "--precision", PRECISION,
    "--seed", str(SEED),
    "--shift", "1.0",
    "--num-inference-steps", "50",
]
modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
if TARGET_LAYERS == "attention + mlp":
    modules += ["gate_proj", "up_proj", "down_proj"]
train_args += ["--target-modules", *modules]
if ADAPTER_TYPE == "lora":
    train_args += ["--rank", str(RANK), "--alpha", str(ALPHA), "--dropout", str(DROPOUT)]
else:
    train_args += ["--lokr-linear-dim", str(RANK), "--lokr-linear-alpha", str(ALPHA)]
if OFFLOAD_ENCODER:
    train_args.append("--offload-encoder")
if RESUME_FROM:
    train_args += ["--resume-from", RESUME_FROM]

TRAIN_CMD = shlex.join(train_args)
print(TRAIN_CMD)

In [ ]:
#@title (Optionnel) TensorBoard — lancer avant l'entraînement pour suivre la loss
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/runs

In [ ]:
#@title 🚀 Lancer l'entraînement
%cd {REPO_DIR}
!{TRAIN_CMD}

## 7. Adaptateur et checkpoints sur le Drive

Tout est écrit directement sur le Drive pendant l'entraînement, rien n'est à copier :

- checkpoints intermédiaires (tous les `SAVE_EVERY` époques, utilisables en inférence) :
  `MyDrive/ACE-Step/lora_output/<RUN_NAME>/checkpoints/epoch_N_loss_X/`
- adaptateur final : `MyDrive/ACE-Step/lora_output/<RUN_NAME>/final/`
- courbes TensorBoard : `MyDrive/ACE-Step/lora_output/<RUN_NAME>/runs/`

Si la session Colab coupe, relancez les sections 1 à 3, puis l'entraînement avec
`RESUME_FROM` = le dernier checkpoint listé ci-dessous.

In [ ]:
#@title Lister les checkpoints sauvegardés et forcer l'écriture sur le Drive
import os

ckpt_root = os.path.join(OUTPUT_DIR, "checkpoints")
if os.path.isdir(ckpt_root):
    ckpts = sorted(os.listdir(ckpt_root), key=lambda n: int(n.split("_")[1]) if n.split("_")[1].isdigit() else 0)
    for name in ckpts:
        print("  ", os.path.join(ckpt_root, name))
    if ckpts:
        print("\nDernier checkpoint (pour RESUME_FROM) :", os.path.join(ckpt_root, ckpts[-1]))
else:
    print("Aucun checkpoint pour l'instant dans", ckpt_root)
final_dir = os.path.join(OUTPUT_DIR, "final")
print("Adaptateur final :", final_dir if os.path.isdir(final_dir) else "(pas encore créé)")

# Le Drive se synchronise en arrière-plan : os.sync() pousse les écritures en attente.
os.sync()
print("✅ Écritures envoyées au Drive")

In [ ]:
#@title (Optionnel) Télécharger l'adaptateur final en .zip sur votre ordinateur
import os
import shutil

from google.colab import files

final_dir = os.path.join(OUTPUT_DIR, "final")
if not os.path.isdir(final_dir):
    print("Pas encore de dossier final :", final_dir)
else:
    archive = shutil.make_archive(os.path.join("/content", f"{RUN_NAME}_final"), "zip", final_dir)
    files.download(archive)

## 8. Tester l'adaptateur dans l'interface Gradio d'ACE-Step

L'interface Gradio sert ici seulement à **générer** (pas à entraîner) : elle charge les
adaptateurs Side-Step tels quels (LoRA PEFT ou LoKR `lokr_weights.safetensors`).

1. Lancez la cellule ci-dessous. Elle démarre Gradio avec le modèle **base** et affiche un lien
   public `https://….gradio.live`.
   Au premier lancement, ACE-Step télécharge les composants manquants (LM 5Hz, DiT turbo, ~6 Go).
2. Dans l'interface : *LoRA Adapter* → collez le chemin affiché → *Load LoRA* → cochez *Use LoRA*,
   réglez *LoRA Scale* (1.0 = pleine force).
3. Générez avec **shift = 1.0** et **~50 steps** (réglages du modèle base), en mettant votre
   `CUSTOM_TAG` dans le prompt.

La cellule tourne tant que l'interface est ouverte : l'interrompre (■) arrête Gradio.

In [ ]:
#@title Lancer Gradio (modèle base + lien public)
ADAPTER_TO_TEST = ""  #@param {type:"string"}
#@markdown Vide = adaptateur final du run courant. Sinon, ex. `.../checkpoints/epoch_50`.
INIT_LLM = "auto"  #@param ["auto", "true", "false"]
#@markdown Le LM 5Hz (Thinking / génération de paroles) est optionnel ; `false` économise la VRAM.

%cd {REPO_DIR}
import os
import shlex

adapter = ADAPTER_TO_TEST or os.path.join(OUTPUT_DIR, "final")
print("Chemin à coller dans « LoRA Adapter » :")
print("   ", adapter, "" if os.path.isdir(adapter) else "(⚠️ introuvable pour l'instant)")

# Gradio cherche les poids ici (mêmes checkpoints que l'entraînement)
os.environ["ACESTEP_CHECKPOINTS_DIR"] = CKPT_DIR
gradio_args = [
    "python", "-m", "acestep.acestep_v15_pipeline",
    "--share",
    "--init_service", "true",
    "--config_path", "acestep-v15-base",
    "--backend", "pt",
    "--allowed-path", WORKDIR,
]
if INIT_LLM != "auto":
    gradio_args += ["--init_llm", INIT_LLM]
!{shlex.join(gradio_args)}